# Customer Churn Analysis with AI/ML

**Intern:** Vishal Pal  
**Project:** Telco Customer Churn Analysis with AI/ML  
**Dataset:** IBM Telco Customer Churn Sample Dataset

## Objective
Analyze customer churn patterns, clean and transform the data, visualize important business patterns, and build machine-learning models that estimate the probability that a customer will churn.

> **Important:** This is an educational analytics project. The model is intended for learning and prioritization, not for making automated decisions about individual customers.


## 1. Business Problem

Customer churn can reduce recurring revenue and increase the cost of acquiring replacement customers. The goal of this project is to answer:

1. What proportion of customers churn?
2. Which customer and service characteristics are associated with churn?
3. How do contract type, tenure, internet service, and monthly charges relate to churn?
4. Can a machine-learning model estimate churn risk?
5. How can model output be translated into practical, data-driven retention insights?


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


## 2. Load the Dataset

The notebook downloads the public IBM sample dataset directly from IBM's archived GitHub repository. If internet access is unavailable, place `Telco-Customer-Churn.csv` in the same folder as this notebook and the local file will be used.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
LOCAL_FILE = "Telco-Customer-Churn.csv"

try:
    df = pd.read_csv(DATA_URL)
    print("Loaded dataset from IBM public source.")
except Exception:
    df = pd.read_csv(LOCAL_FILE)
    print("Loaded local Telco-Customer-Churn.csv.")

print("Shape:", df.shape)
display(df.head())


## 3. Data Understanding and Quality Checks

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.info())

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).head(10))

print("\nDuplicate customer IDs:", df["customerID"].duplicated().sum())
display(df["Churn"].value_counts())
display(df["Churn"].value_counts(normalize=True).mul(100).round(2))


## 4. Data Cleaning and Feature Engineering

`TotalCharges` is stored as text in the IBM sample. We convert it to numeric values, remove invalid/missing records where necessary, encode the target, and create a small number of interpretable derived features.


In [ ]:
df = df.copy()

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remove rows with missing target and fill numeric missing values later in the pipeline.
df = df.dropna(subset=["Churn"])

# Binary target
df["ChurnFlag"] = df["Churn"].map({"Yes": 1, "No": 0})

# Useful business features
service_cols = [
    "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
]
existing_service_cols = [c for c in service_cols if c in df.columns]
df["ServiceCount"] = sum(
    (df[c].eq("Yes")).astype(int) for c in existing_service_cols
)

df["TenureBand"] = pd.cut(
    df["tenure"],
    bins=[-1, 6, 12, 24, 48, 72],
    labels=["0-6", "7-12", "13-24", "25-48", "49-72"]
)

print("Cleaned shape:", df.shape)
display(df.head())


## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Churn distribution
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="Churn")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Customers")
plt.show()

# Churn by contract
contract_churn = pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100
display(contract_churn.round(2))

plt.figure(figsize=(8,5))
sns.barplot(data=df, x="Contract", y="ChurnFlag", estimator=np.mean)
plt.title("Average Churn Rate by Contract Type")
plt.ylabel("Churn Rate")
plt.xlabel("Contract")
plt.gca().yaxis.set_major_formatter(
    plt.FuncFormatter(lambda y, _: f"{y:.0%}")
)
plt.show()


In [ ]:
# Churn by tenure band
tenure_churn = df.groupby("TenureBand", observed=False)["ChurnFlag"].mean().mul(100).round(2)
display(tenure_churn.to_frame("ChurnRatePercent"))

plt.figure(figsize=(8,5))
sns.barplot(x=tenure_churn.index.astype(str), y=tenure_churn.values)
plt.title("Churn Rate by Tenure Band")
plt.xlabel("Tenure (months)")
plt.ylabel("Churn Rate (%)")
plt.show()

# Monthly charges
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges by Churn Status")
plt.show()


In [ ]:
# Internet service
internet_churn = pd.crosstab(df["InternetService"], df["Churn"], normalize="index") * 100
display(internet_churn.round(2))

plt.figure(figsize=(8,5))
sns.barplot(data=df, x="InternetService", y="ChurnFlag", estimator=np.mean)
plt.title("Churn Rate by Internet Service")
plt.ylabel("Churn Rate")
plt.xlabel("Internet Service")
plt.gca().yaxis.set_major_formatter(
    plt.FuncFormatter(lambda y, _: f"{y:.0%}")
)
plt.show()


## 6. Statistical Summary

We compare numerical variables across churn groups to identify meaningful differences. This is descriptive analysis; correlation or group differences do not prove that a factor causes churn.


In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges", "ServiceCount"]
summary = df.groupby("Churn")[numeric_cols].agg(["mean", "median"]).round(2)
display(summary)


## 7. Machine Learning Preparation

We exclude the customer ID and the original text target from the model features. Categorical variables are one-hot encoded and numeric variables are imputed and standardized where appropriate.


In [ ]:
target = df["ChurnFlag"]

drop_cols = ["customerID", "Churn", "ChurnFlag"]
X = df.drop(columns=[c for c in drop_cols if c in df.columns])

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, target, test_size=0.20, random_state=42, stratify=target
)

print("Train:", X_train.shape, "Test:", X_test.shape)


## 8. Model 1 — Logistic Regression

Logistic regression provides an interpretable baseline for binary classification.


In [ ]:
logistic_model = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

logistic_model.fit(X_train, y_train)
log_pred = logistic_model.predict(X_test)
log_prob = logistic_model.predict_proba(X_test)[:, 1]

log_metrics = {
    "Accuracy": accuracy_score(y_test, log_pred),
    "Precision": precision_score(y_test, log_pred),
    "Recall": recall_score(y_test, log_pred),
    "F1": f1_score(y_test, log_pred),
    "ROC-AUC": roc_auc_score(y_test, log_prob)
}
display(pd.DataFrame(log_metrics, index=["Logistic Regression"]).round(4))
print(classification_report(y_test, log_pred))


## 9. Model 2 — Random Forest

Random Forest can capture non-linear relationships and interactions between variables.


In [ ]:
rf_model = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

rf_metrics = {
    "Accuracy": accuracy_score(y_test, rf_pred),
    "Precision": precision_score(y_test, rf_pred),
    "Recall": recall_score(y_test, rf_pred),
    "F1": f1_score(y_test, rf_pred),
    "ROC-AUC": roc_auc_score(y_test, rf_prob)
}
display(pd.DataFrame(rf_metrics, index=["Random Forest"]).round(4))
print(classification_report(y_test, rf_pred))


## 10. Compare Models

Use ROC-AUC together with precision, recall and F1 rather than relying only on accuracy, because churn datasets are commonly imbalanced.


In [ ]:
comparison = pd.DataFrame([log_metrics, rf_metrics], index=["Logistic Regression", "Random Forest"])
display(comparison.round(4))

plt.figure(figsize=(8,6))
RocCurveDisplay.from_predictions(y_test, log_prob, name="Logistic Regression")
RocCurveDisplay.from_predictions(y_test, rf_prob, name="Random Forest")
plt.title("ROC Curves")
plt.show()


## 11. Confusion Matrix

In [ ]:
best_pred = rf_pred
cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


## 12. Feature Importance / Explainability

For the Random Forest, we extract feature importance after preprocessing. This helps explain which transformed variables contribute most to the model's predictions. Feature importance should be interpreted as association within the model, not causal impact.


In [ ]:
prep_fitted = rf_model.named_steps["prep"]
rf_estimator = rf_model.named_steps["model"]

feature_names = prep_fitted.get_feature_names_out()
importances = rf_estimator.feature_importances_

importance_df = (
    pd.DataFrame({"Feature": feature_names, "Importance": importances})
    .sort_values("Importance", ascending=False)
    .head(15)
)

display(importance_df)

plt.figure(figsize=(10,6))
sns.barplot(data=importance_df, x="Importance", y="Feature")
plt.title("Top 15 Random Forest Feature Importances")
plt.show()


## 13. AI/ML Risk Scoring

The model probability is converted into simple educational risk bands. This is a prioritization aid, not an automatic decision system.


In [ ]:
risk_output = X_test.copy()
risk_output["ActualChurn"] = y_test.values
risk_output["PredictedChurnProbability"] = rf_prob

risk_output["RiskBand"] = pd.cut(
    risk_output["PredictedChurnProbability"],
    bins=[-0.01, 0.30, 0.60, 1.01],
    labels=["Low", "Medium", "High"]
)

display(
    risk_output[["tenure", "Contract", "MonthlyCharges",
                 "PredictedChurnProbability", "RiskBand"]]
    .sort_values("PredictedChurnProbability", ascending=False)
    .head(20)
)


## 14. AI-Assisted Business Insights

The following rules turn the analytical findings into concise, human-readable observations. In a production system, this layer could be connected to an approved generative-AI service to draft stakeholder summaries, while keeping the underlying metrics traceable to the dataset and model.


In [ ]:
overall_churn = df["ChurnFlag"].mean()
month_to_month = df.loc[df["Contract"] == "Month-to-month", "ChurnFlag"].mean()
one_year = df.loc[df["Contract"] == "One year", "ChurnFlag"].mean()
two_year = df.loc[df["Contract"] == "Two year", "ChurnFlag"].mean()
short_tenure = df.loc[df["tenure"] <= 12, "ChurnFlag"].mean()

insights = [
    f"Overall churn rate in the sample is approximately {overall_churn:.1%}.",
    f"Month-to-month customers show a churn rate of approximately {month_to_month:.1%}.",
    f"One-year customers show a churn rate of approximately {one_year:.1%}.",
    f"Two-year customers show a churn rate of approximately {two_year:.1%}.",
    f"Customers with tenure of 12 months or less show a churn rate of approximately {short_tenure:.1%}.",
    "Use these patterns to prioritize retention analysis, but do not interpret them as proof that contract or tenure causes churn."
]

for i, item in enumerate(insights, 1):
    print(f"{i}. {item}")


## 15. Business Recommendations

- Monitor new customers during the first year because the EDA can reveal elevated early-tenure churn.
- Examine month-to-month customers separately when designing retention experiments.
- Review high monthly-charge customers together with their subscribed services and contract type.
- Use model probabilities to prioritize which customer groups deserve further investigation.
- Validate any retention action with controlled experiments before claiming causal impact.
- Protect customer privacy and avoid using the model for unfair or high-impact automated decisions.

## Limitations

- The IBM dataset is a sample snapshot rather than a live production database.
- The data does not provide a full monthly time series for each customer.
- Association in the dataset does not establish causation.
- Model performance depends on the train/test split and preprocessing choices.
- Risk scores should support human analysis rather than replace human judgment.

## Conclusion

This project demonstrates a complete data-analytics workflow: data loading, cleaning, exploratory analysis, feature engineering, visualization, machine learning, model evaluation, explainability, and AI-assisted business insight generation.
